In [1]:
import pandas as pd
import numpy as np

df_xlsx = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df_xlsx.shape}")
print(f"\nColumn types:\n{df_xlsx.dtypes}")
print(f"\nMissing values:\n{df_xlsx.isnull().sum()}")
print(f"\nFirst rows:\n{df_xlsx.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

In [2]:
#________________TASK 1.1________________

In [3]:
retail_csv = df_xlsx.to_csv("data.csv")

df = pd.read_csv("data.csv")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

num_rows, num_cols = df.shape
print(f"Rows: {num_rows}")
print(f"Columns: {num_cols}")

memory_usage = df.memory_usage(deep=True).sum() / (1024**2)
print(f"Total Memory Usage: {memory_usage:.2f} MB")

missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent
})

print("\nMissing Values Summary:")
print(missing_summary)

unique_values = df.nunique()

print("\nUnique Values Per Column:")
print(unique_values)

numeric_stats = df.describe().T
numeric_stats["median"] = df.median(numeric_only=True)

print("\nNumeric Column Statistics:")
print(numeric_stats[["min", "max", "mean", "median", "std"]])

Rows: 541909
Columns: 9
Total Memory Usage: 148.33 MB

Missing Values Summary:
             Missing Count  Missing %
Unnamed: 0               0   0.000000
InvoiceNo                0   0.000000
StockCode                0   0.000000
Description           1454   0.268311
Quantity                 0   0.000000
InvoiceDate              0   0.000000
UnitPrice                0   0.000000
CustomerID          135080  24.926694
Country                  0   0.000000

Unique Values Per Column:
Unnamed: 0     541909
InvoiceNo       25900
StockCode        4070
Description      4223
Quantity          722
InvoiceDate     23260
UnitPrice        1630
CustomerID       4372
Country            38
dtype: int64

Numeric Column Statistics:
                             min                  max  \
Unnamed: 0                   0.0             541908.0   
Quantity                -80995.0              80995.0   
InvoiceDate  2010-12-01 08:26:00  2011-12-09 12:50:00   
UnitPrice              -11062.06              3

In [4]:
#________________TASK 1.2________________

In [5]:
"""
The column CustomerID has many missing values. The missing data is probably MAR (Missing At Random). 
This means the missing values are related to the situation of the purchase. 
For example, some customers buy items as guests and do not register in the system, so their CustomerID is not recorded. 
In these rows, other information like InvoiceNo, Quantity, UnitPrice, and InvoiceDate is still available. 
Because CustomerID is important for customer analysis, it is not good to create fake IDs. Fake IDs would create incorrect customers in the data. 
Therefore, the best solution is to remove the rows where CustomerID is missing.

The column Description also has missing values. In this case, the missing data is probably MCAR (Missing Completely At Random). 
The missing descriptions appear randomly in the dataset and do not seem connected to other columns like Quantity or UnitPrice. 
Also, the product can still be identified by the StockCode column. Because Description is only text that explains the product 
removing these rows would delete useful sales data. A better solution is to fill the missing values with a simple label such as "Unknown" or "Missing Description". 
This keeps the data while showing that the description is not available.
"""

"""
First, we look at the CustomerID column. We compare transactions with a missing CustomerID and transactions where CustomerID is present. 
We check other columns such as Country, Quantity, and UnitPrice. 
The results usually show that the distributions of Quantity and UnitPrice are similar in both groups. 
The Country values are also similar, which means the transactions happen in the same places. 
This suggests that the purchases themselves are normal transactions. 
The difference is that the customer was not identified in the system, which can happen with guest purchases. 
Because the missing value is related to how the transaction was recorded and not random noise, the missingness is likely MAR (Missing At Random).

Next, we look at the Description column. 
We check whether rows with missing Description still have valid StockCode values. In most cases, these rows do have a valid StockCode. 
We can also look at other rows with the same StockCode and see that they usually have a description for the same product. 
This means the product information exists elsewhere in the dataset. 
Because the missing description does not change other values like Quantity or UnitPrice and can often be recovered using StockCode, the missingness appears random and is likely MCAR (Missing Completely At Random).
"""

'\nFirst, we look at the CustomerID column. We compare transactions with a missing CustomerID and transactions where CustomerID is present. \nWe check other columns such as Country, Quantity, and UnitPrice. \nThe results usually show that the distributions of Quantity and UnitPrice are similar in both groups. \nThe Country values are also similar, which means the transactions happen in the same places. \nThis suggests that the purchases themselves are normal transactions. \nThe difference is that the customer was not identified in the system, which can happen with guest purchases. \nBecause the missing value is related to how the transaction was recorded and not random noise, the missingness is likely MAR (Missing At Random).\n\nNext, we look at the Description column. \nWe check whether rows with missing Description still have valid StockCode values. In most cases, these rows do have a valid StockCode. \nWe can also look at other rows with the same StockCode and see that they usually 

In [6]:
#________________TASK 1.3________________

In [7]:
# Check missing values before handling
print(df.isnull().sum())

# Drop rows with missing Description
# Description is important for identifying the product; can’t leave it missing
df = df.dropna(subset=["Description"])

# Compute the most common CustomerID per Country
# Will be used to fill missing CustomerIDs to preserve data
most_common_id = df.groupby("Country")["CustomerID"].agg(
    lambda x: x.value_counts().idxmax() if not x.value_counts().empty else np.nan
)

# Fill missing CustomerID using the most common ID for that Country
# This prevents losing ~25% of rows and keeps customer data reasonable
df["CustomerID"] = df["CustomerID"].fillna(df["Country"].map(most_common_id))

# Fill remaining missing CustomerIDs in Hong Kong with a default ID
# Ensures no CustomerID remains missing
df.loc[(df["CustomerID"].isnull()) & (df["Country"] == "Hong Kong"), "CustomerID"] = 8

# Check missing values after handling CustomerID
print(df.isnull().sum())

# Fill missing Description with "Unknown"
# Keeps all transactions while marking missing textual info
df['Description'] = df['Description'].fillna("Unknown")

# Final check to confirm no missing values remain
missing_summary = df.isnull().sum()
print("Remaining missing values per column:")
print(missing_summary)

Unnamed: 0          0
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Unnamed: 0     0
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64
Remaining missing values per column:
Unnamed: 0     0
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [8]:
#________________TASK 2.1________________

In [9]:
cancellations = df[df["InvoiceNo"].str.startswith("C")]
num_cancellations = len(cancellations)
print(f"Number of cancellation transactions: {num_cancellations}")

df["IsCancellation"] = df["InvoiceNo"].str.startswith("C")

"""
Cancellations, which are invoices starting with "C", represent returns and have negative quantities. 
These transactions do not reflect actual purchases. 
For predicting customer churn, cancellations are important because frequent returns may indicate that a customer is dissatisfied or at risk of leaving. 
By keeping cancellations and flagging them, the model can learn patterns related to unhappy or disengaged customers without confusing returns with normal purchases. 
For building a recommender system, cancellations should not be counted as positive purchases. 
Flagging them allows the system to ignore or give negative weight to these transactions while preserving the information, ensuring that recommendations are based on real customer interest rather than returned items. 
In this way, flagging cancellations retains useful behavioral data while preventing distortion in sales-based models.
"""

Number of cancellation transactions: 9288


'\nCancellations, which are invoices starting with "C", represent returns and have negative quantities. \nThese transactions do not reflect actual purchases. \nFor predicting customer churn, cancellations are important because frequent returns may indicate that a customer is dissatisfied or at risk of leaving. \nBy keeping cancellations and flagging them, the model can learn patterns related to unhappy or disengaged customers without confusing returns with normal purchases. \nFor building a recommender system, cancellations should not be counted as positive purchases. \nFlagging them allows the system to ignore or give negative weight to these transactions while preserving the information, ensuring that recommendations are based on real customer interest rather than returned items. \nIn this way, flagging cancellations retains useful behavioral data while preventing distortion in sales-based models.\n'

In [10]:
#________________TASK 2.2________________

In [11]:
invalid_conditions = {
    "Negative or zero quantity (not cancellations)": (df["Quantity"] <= 0) & (~df["InvoiceNo"].str.startswith("C")),
    "Zero or negative UnitPrice": df["UnitPrice"] <= 0,
    "Extreme Quantity outliers": (df["Quantity"] > df["Quantity"].quantile(0.99)) | (df["Quantity"] < df["Quantity"].quantile(0.01)),
    "Extreme UnitPrice outliers": (df["UnitPrice"] > df["UnitPrice"].quantile(0.99)) | (df["UnitPrice"] < df["UnitPrice"].quantile(0.01))
}

for name, condition in invalid_conditions.items():
    count = condition.sum()
    print(f"{name}: {count} records")
    if "outliers" in name:
        df["Flag_" + name.replace(" ", "_")] = condition
    else:
        df = df[~condition]

print("Remaining data shape:", df.shape)
print("Remaining missing values:", df.isnull().sum())

"""
1. Negative or zero Quantity (that are not cancellations):  
These rows do not represent valid purchases. Negative quantities usually indicate data errors, and zero quantity does not contribute to sales. 
Therefore, we remove these rows to keep only valid transactions in the dataset.

2. Zero or negative UnitPrice:  
Prices of zero or less are invalid and likely data entry errors. Keeping these rows would distort revenue and analysis. 
Therefore, we remove these rows from the dataset.

3. Extreme Quantity outliers:  
Very large or very small quantities can distort statistical analysis and models. 
Instead of removing them, we flag these rows so that downstream models can handle them separately or consider capping their effect.

4. Extreme UnitPrice outliers:  
Similarly, extremely high or low prices can skew results and create bias in predictive models. 
We flag these rows to identify them while preserving the data for analysis, allowing special treatment if needed.
"""

Negative or zero quantity (not cancellations): 474 records
Zero or negative UnitPrice: 1063 records
Extreme Quantity outliers: 9203 records
Extreme UnitPrice outliers: 10164 records
Remaining data shape: (539392, 12)
Remaining missing values: Unnamed: 0                         0
InvoiceNo                          0
StockCode                          0
Description                        0
Quantity                           0
InvoiceDate                        0
UnitPrice                          0
CustomerID                         0
Country                            0
IsCancellation                     0
Flag_Extreme_Quantity_outliers     0
Flag_Extreme_UnitPrice_outliers    0
dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_1420\135809138.py:14: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df = df[~condition]


'\n1. Negative or zero Quantity (that are not cancellations):  \nThese rows do not represent valid purchases. Negative quantities usually indicate data errors, and zero quantity does not contribute to sales. \nTherefore, we remove these rows to keep only valid transactions in the dataset.\n\n2. Zero or negative UnitPrice:  \nPrices of zero or less are invalid and likely data entry errors. Keeping these rows would distort revenue and analysis. \nTherefore, we remove these rows from the dataset.\n\n3. Extreme Quantity outliers:  \nVery large or very small quantities can distort statistical analysis and models. \nInstead of removing them, we flag these rows so that downstream models can handle them separately or consider capping their effect.\n\n4. Extreme UnitPrice outliers:  \nSimilarly, extremely high or low prices can skew results and create bias in predictive models. \nWe flag these rows to identify them while preserving the data for analysis, allowing special treatment if needed.\n'

In [12]:
#________________TASK 2.3________________

In [13]:
df["IsCancellation"] = df["InvoiceNo"].str.startswith("C")

invalid_qty = (df["Quantity"] <= 0) & (~df["IsCancellation"])

invalid_price = df["UnitPrice"] <= 0

df_cleaned = df[~(invalid_qty | invalid_price)].copy()

q_low = df_cleaned["Quantity"].quantile(0.01)
q_high = df_cleaned["Quantity"].quantile(0.99)
df_cleaned["Flag_Quantity_Outlier"] = (df_cleaned["Quantity"] < q_low) | (df_cleaned["Quantity"] > q_high)

p_low = df_cleaned["UnitPrice"].quantile(0.01)
p_high = df_cleaned["UnitPrice"].quantile(0.99)
df_cleaned["Flag_UnitPrice_Outlier"] = (df_cleaned["UnitPrice"] < p_low) | (df_cleaned["UnitPrice"] > p_high)

neg_qty_remaining = df_cleaned[(df_cleaned["Quantity"] <= 0) & (~df_cleaned["IsCancellation"])]
print(f"Negative or zero quantities (non-cancellations) remaining: {len(neg_qty_remaining)}")

neg_price_remaining = df_cleaned[df_cleaned["UnitPrice"] <= 0]
print(f"Zero or negative UnitPrice remaining: {len(neg_price_remaining)}")

print(f"Original data shape: {df.shape}")
print(f"Cleaned data shape: {df_cleaned.shape}")

print("\nSample of flagged extreme Quantity or UnitPrice rows:")
print(df_cleaned[df_cleaned["Flag_Quantity_Outlier"] | df_cleaned["Flag_UnitPrice_Outlier"]].head())

Negative or zero quantities (non-cancellations) remaining: 0
Zero or negative UnitPrice remaining: 0
Original data shape: (539392, 12)
Cleaned data shape: (539392, 14)

Sample of flagged extreme Quantity or UnitPrice rows:
     Unnamed: 0 InvoiceNo StockCode                       Description  \
96           96    536378     21212   PACK OF 72 RETROSPOT CAKE CASES   
141         141   C536379         D                          Discount   
168         168    536385     22783  SET 3 WICKER OVAL BASKETS W LIDS   
178         178    536387     79321                     CHILLI LIGHTS   
179         179    536387     22780    LIGHT GARLAND BUTTERFILES PINK   

     Quantity         InvoiceDate  UnitPrice  CustomerID         Country  \
96        120 2010-12-01 09:37:00       0.42     14688.0  United Kingdom   
141        -1 2010-12-01 09:41:00      27.50     14527.0  United Kingdom   
168         1 2010-12-01 09:56:00      19.95     17420.0  United Kingdom   
178       192 2010-12-01 09:58:00 

In [14]:
#________________TASK 3.1________________

In [15]:
df_cleaned["Country"].unique()

country_counts = df_cleaned["Country"].value_counts()

top5_countries = country_counts.head(5)
top5_percentage = top5_countries.sum() / len(df_cleaned) * 100
print(f"Top 5 countries:\n{top5_countries}")
print(f"Percentage of transactions from top 5 countries: {top5_percentage:.2f}%")

few_tx_countries = (country_counts < 50).sum()
print(f"Number of countries with fewer than 50 transactions: {few_tx_countries}")

rare_countries = country_counts[country_counts < 50].index.tolist()

df_cleaned["Country_Cleaned"] = df_cleaned["Country"].apply(lambda x: "Other" if x in rare_countries else x)

num_categories_before = df_cleaned["Country"].nunique()
num_categories_after = df_cleaned["Country_Cleaned"].nunique()

print(f"Number of country categories before cleaning: {num_categories_before}")
print(f"Number of country categories after grouping rare countries into 'Other': {num_categories_after}")

print("\nTransaction counts after cleaning:")
print(df_cleaned["Country_Cleaned"].value_counts())

Top 5 countries:
Country
United Kingdom    492979
Germany             9493
France              8556
EIRE                8192
Spain               2532
Name: count, dtype: int64
Percentage of transactions from top 5 countries: 96.73%
Number of countries with fewer than 50 transactions: 6
Number of country categories before cleaning: 38
Number of country categories after grouping rare countries into 'Other': 33

Transaction counts after cleaning:
Country_Cleaned
United Kingdom          492979
Germany                   9493
France                    8556
EIRE                      8192
Spain                     2532
Netherlands               2367
Belgium                   2069
Switzerland               2001
Portugal                  1519
Australia                 1256
Norway                    1085
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria  

In [16]:
#________________TASK 3.2________________

In [17]:
unique_stockcodes = df_cleaned["StockCode"].nunique()
print("Number of unique StockCodes:", unique_stockcodes)

non_product_codes = df_cleaned[~df_cleaned["StockCode"].str.isnumeric()]["StockCode"].unique()

print("Non-product StockCodes:")
print(non_product_codes)

non_product_rows = df_cleaned[~df_cleaned["StockCode"].str.isnumeric()]
print("Number of non-product transactions:", len(non_product_rows))

df_products = df_cleaned[df_cleaned["StockCode"].str.isnumeric()].copy()

print("Shape after removing non-product codes:", df_products.shape)

Number of unique StockCodes: 3938
Non-product StockCodes:
['85123A' '84406B' '84029G' ... '85179a' '90214U' '47591b']
Number of non-product transactions: 54245
Shape after removing non-product codes: (485147, 15)


In [18]:
#________________TASK 3.3________________

In [19]:
# Convert descriptions to uppercase to standardize matching
df_cleaned["Description_upper"] = df_cleaned["Description"].str.upper()

df_cleaned["Has_SET"] = df_cleaned["Description_upper"].str.contains("SET", na=False)
df_cleaned["Has_PACK"] = df_cleaned["Description_upper"].str.contains("PACK", na=False)
df_cleaned["Has_VINTAGE"] = df_cleaned["Description_upper"].str.contains("VINTAGE", na=False)

print("Average Quantity by SET keyword:")
print(df_cleaned.groupby("Has_SET")["Quantity"].mean())

print("\nAverage Quantity by PACK keyword:")
print(df_cleaned.groupby("Has_PACK")["Quantity"].mean())

print("\nAverage Quantity by VINTAGE keyword:")
print(df_cleaned.groupby("Has_VINTAGE")["Quantity"].mean())

print("\nAverage UnitPrice by SET keyword:")
print(df_cleaned.groupby("Has_SET")["UnitPrice"].mean())

print("\nAverage UnitPrice by PACK keyword:")
print(df_cleaned.groupby("Has_PACK")["UnitPrice"].mean())

print("\nAverage UnitPrice by VINTAGE keyword:")
print(df_cleaned.groupby("Has_VINTAGE")["UnitPrice"].mean())

df_cleaned["Description_WordCount"] = df_cleaned["Description"].str.split().apply(len)

print(df_cleaned[["Description_WordCount", "Quantity", "UnitPrice"]].corr())

Average Quantity by SET keyword:
Has_SET
False    10.016485
True      8.592117
Name: Quantity, dtype: float64

Average Quantity by PACK keyword:
Has_PACK
False     9.614583
True     17.177041
Name: Quantity, dtype: float64

Average Quantity by VINTAGE keyword:
Has_VINTAGE
False     9.829498
True     10.091888
Name: Quantity, dtype: float64

Average UnitPrice by SET keyword:
Has_SET
False    4.834893
True     3.488473
Name: UnitPrice, dtype: float64

Average UnitPrice by PACK keyword:
Has_PACK
False    4.785280
True     1.135756
Name: UnitPrice, dtype: float64

Average UnitPrice by VINTAGE keyword:
Has_VINTAGE
False    4.736453
True     3.731984
Name: UnitPrice, dtype: float64
                       Description_WordCount  Quantity  UnitPrice
Description_WordCount               1.000000 -0.000705  -0.037207
Quantity                           -0.000705  1.000000  -0.001369
UnitPrice                          -0.037207 -0.001369   1.000000


In [20]:
#________________TASK 4.1________________

In [21]:
df_products["Revenue"] = df_products["Quantity"] * df_products["UnitPrice"]
customer_df = df_products.groupby("CustomerID").agg({
    "Revenue": "sum",
    "InvoiceNo": "nunique",
    "StockCode": "nunique",
    "InvoiceDate": ["min", "max"]
})
customer_df.columns = [
    "TotalRevenue",
    "NumOrders",
    "NumDistinctProducts",
    "FirstPurchaseDate",
    "LastPurchaseDate"
]

customer_df = customer_df.reset_index()

customer_df["CustomerLifetimeDays"] = (
    customer_df["LastPurchaseDate"] - customer_df["FirstPurchaseDate"]
).dt.days

threshold = customer_df["TotalRevenue"].quantile(0.90)

customer_df["HighValueCustomer"] = (
    customer_df["TotalRevenue"] >= threshold
).astype(int)

print(customer_df["HighValueCustomer"].value_counts())

print(customer_df.head())
print(customer_df.shape)

HighValueCustomer
0    3907
1     435
Name: count, dtype: int64
   CustomerID  TotalRevenue  NumOrders  NumDistinctProducts  \
0         8.0       8527.25          8                  172   
1     12346.0          0.00          2                    1   
2     12347.0       3653.45          7                   90   
3     12348.0       1437.24          4                   21   
4     12349.0       1372.42          1                   68   

    FirstPurchaseDate    LastPurchaseDate  CustomerLifetimeDays  \
0 2011-01-24 14:24:00 2011-10-28 08:20:00                   276   
1 2011-01-18 10:01:00 2011-01-18 10:17:00                     0   
2 2010-12-07 14:57:00 2011-12-07 15:52:00                   365   
3 2010-12-16 19:09:00 2011-09-25 13:13:00                   282   
4 2011-11-21 09:51:00 2011-11-21 09:51:00                     0   

   HighValueCustomer  
0                  1  
1                  0  
2                  1  
3                  0  
4                  0  
(4342, 8)


In [22]:
#________________TASK 4.2________________

In [23]:
"""
Most customers are regular (0) and only about 10% are high-value (1), so the dataset is imbalanced.

If a model always predicts "regular", it would still get about 90% accuracy, because most customers are regular.

Accuracy is a poor metric here because the model would never detect high-value customers.
"""

'\nMost customers are regular (0) and only about 10% are high-value (1), so the dataset is imbalanced.\n\nIf a model always predicts "regular", it would still get about 90% accuracy, because most customers are regular.\n\nAccuracy is a poor metric here because the model would never detect high-value customers.\n'

In [24]:
#class_counts = customer_df["HighValueCustomer"].value_counts()

#print(class_counts)

#class_percent = customer_df["HighValueCustomer"].value_counts(normalize=True) * 100
#print(class_percent)

In [25]:
#________________TASK 4.3________________

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

X = customer_df.drop(columns=[
    "HighValueCustomer",
    "CustomerID",
    "FirstPurchaseDate",
    "LastPurchaseDate",
    "TotalRevenue"
])

y = customer_df["HighValueCustomer"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train distribution:")
print(y_train.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

train_data = X_train.copy()
train_data["target"] = y_train

majority = train_data[train_data["target"] == 0]
minority = train_data[train_data["target"] == 1]

print("Before oversampling:")
print(train_data["target"].value_counts())

minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled = pd.concat([majority, minority_oversampled])

print("\nAfter oversampling:")
print(oversampled["target"].value_counts())

X_train_over = oversampled.drop(columns="target")
y_train_over = oversampled["target"]

majority_undersampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_undersampled, minority])

print("\nAfter undersampling:")
print(undersampled["target"].value_counts())

X_train_under = undersampled.drop(columns="target")
y_train_under = undersampled["target"]

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Original training data")
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

model_over = LogisticRegression(max_iter=1000)

model_over.fit(X_train_over, y_train_over)

pred_over = model_over.predict(X_test)

print("\nOversampled training data")
print("Precision:", precision_score(y_test, pred_over))
print("Recall:", recall_score(y_test, pred_over))
print("F1:", f1_score(y_test, pred_over))

model_under = LogisticRegression(max_iter=1000)

model_under.fit(X_train_under, y_train_under)

pred_under = model_under.predict(X_test)

print("\nUndersampled training data")
print("Precision:", precision_score(y_test, pred_under))
print("Recall:", recall_score(y_test, pred_under))
print("F1:", f1_score(y_test, pred_under))

"""
The training dataset is imbalanced (3125 regular vs 348 high-value customers).

After oversampling, both classes have 3125 samples.  
After undersampling, both classes have 348 samples.

Model results on the test set:

Original data:
Precision = 0.85  
Recall = 0.60  
F1 = 0.70  

Oversampling:
Precision = 0.51  
Recall = 0.87  
F1 = 0.64  

Undersampling:
Precision = 0.50  
Recall = 0.87  
F1 = 0.64  

Oversampling and undersampling improve recall, so the model finds more high-value customers.  
However, precision becomes lower.
"""

Train distribution:
HighValueCustomer
0    3125
1     348
Name: count, dtype: int64

Test distribution:
HighValueCustomer
0    782
1     87
Name: count, dtype: int64
Before oversampling:
target
0    3125
1     348
Name: count, dtype: int64

After oversampling:
target
0    3125
1    3125
Name: count, dtype: int64

After undersampling:
target
0    348
1    348
Name: count, dtype: int64
Original training data
Precision: 0.8524590163934426
Recall: 0.5977011494252874
F1: 0.7027027027027027

Oversampled training data
Precision: 0.5100671140939598
Recall: 0.8735632183908046
F1: 0.6440677966101694

Undersampled training data
Precision: 0.5033112582781457
Recall: 0.8735632183908046
F1: 0.6386554621848739


'\nThe training dataset is imbalanced (3125 regular vs 348 high-value customers).\n\nAfter oversampling, both classes have 3125 samples.  \nAfter undersampling, both classes have 348 samples.\n\nModel results on the test set:\n\nOriginal data:\nPrecision = 0.85  \nRecall = 0.60  \nF1 = 0.70  \n\nOversampling:\nPrecision = 0.51  \nRecall = 0.87  \nF1 = 0.64  \n\nUndersampling:\nPrecision = 0.50  \nRecall = 0.87  \nF1 = 0.64  \n\nOversampling and undersampling improve recall, so the model finds more high-value customers.  \nHowever, precision becomes lower.\n'

In [27]:
#________________TASK 5.1________________

In [28]:
df_dec = df[df["InvoiceDate"].between("2011-12-01", "2011-12-31")]

customer_target = df_dec.groupby("CustomerID")["InvoiceNo"].count().reset_index()
customer_target["MadePurchaseDec2011"] = 1

all_customers = pd.DataFrame({"CustomerID": df["CustomerID"].unique()})
customer_target = all_customers.merge(customer_target, on="CustomerID", how="left").fillna(0)
customer_target["MadePurchaseDec2011"] = customer_target["MadePurchaseDec2011"].astype(int)

df["Revenue"] = df["Quantity"] * df["UnitPrice"]

customer_features = df.groupby("CustomerID").agg({
    "Quantity": "sum",
    "UnitPrice": "mean",
    "Revenue": "sum"
}).reset_index()

customer_full = customer_features.merge(customer_target, on="CustomerID")

X = customer_full.drop(columns=["MadePurchaseDec2011"])
y = customer_full["MadePurchaseDec2011"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Leaked model performance:")
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

print("Feature period ends:", df["InvoiceDate"].max())
print("Target period starts:", pd.Timestamp("2011-12-01"))

corrs = customer_full.corr()["MadePurchaseDec2011"].sort_values(ascending=False)
print(corrs)

observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]

df_pred = df[df["InvoiceDate"] >= prediction_start]

features_obs = df_obs.groupby("CustomerID").agg({
    "Quantity": "sum",
    "UnitPrice": "mean",
    "Revenue": "sum"
}).reset_index()

target_pred = df_pred.groupby("CustomerID")["InvoiceNo"].nunique().reset_index()
target_pred["MadePurchase"] = (target_pred["InvoiceNo"] > 0).astype(int)
target_pred = target_pred[["CustomerID", "MadePurchase"]]

customer_temporal = features_obs.merge(target_pred, on="CustomerID", how="left").fillna(0)

X_temp = customer_temporal.drop(columns=["CustomerID", "MadePurchase"])
y_temp = customer_temporal["MadePurchase"]

X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)

model_temp = LogisticRegression(max_iter=1000)
model_temp.fit(X_train, y_train)
pred_temp = model_temp.predict(X_test)

print("\nCorrect temporal split model performance:")
print("Precision:", precision_score(y_test, pred_temp))
print("Recall:", recall_score(y_test, pred_temp))
print("F1:", f1_score(y_test, pred_temp))

"""
Features computed on the full time range included data from the target month.  
Model performance is suspiciously high (Precision, Recall, F1 all inflated).  
Feature-target correlation showed some features too predictive (sign of leakage).

Observation window: Dec 2010 — Sep 2011  
Prediction window: Oct 2011 — Dec 2011  
Features computed only from observation window  
Target computed from prediction window

Model performance decreased to realistic levels.  
Temporal split prevents using future information for prediction.  

Always compute features only from the past.  
Random splits with future data cause temporal leakage, giving over-optimistic metrics.
"""

Leaked model performance:
Precision: 1.0
Recall: 1.0
F1: 1.0
Feature period ends: 2011-12-09 12:50:00
Target period starts: 2011-12-01 00:00:00
MadePurchaseDec2011    1.000000
Quantity               0.140300
InvoiceNo              0.107090
Revenue                0.098286
UnitPrice             -0.013961
CustomerID            -0.019600
Name: MadePurchaseDec2011, dtype: float64

Correct temporal split model performance:
Precision: 0.725
Recall: 0.5272727272727272
F1: 0.6105263157894737


'\nFeatures computed on the full time range included data from the target month.  \nModel performance is suspiciously high (Precision, Recall, F1 all inflated).  \nFeature-target correlation showed some features too predictive (sign of leakage).\n\nObservation window: Dec 2010 — Sep 2011  \nPrediction window: Oct 2011 — Dec 2011  \nFeatures computed only from observation window  \nTarget computed from prediction window\n\nModel performance decreased to realistic levels.  \nTemporal split prevents using future information for prediction.  \n\nAlways compute features only from the past.  \nRandom splits with future data cause temporal leakage, giving over-optimistic metrics.\n'

In [29]:
#________________TASK 5.2________________

In [30]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
corr_df = X.copy()
corr_df["target"] = y

print(corr_df.corr(numeric_only=True)["target"].sort_values(ascending=False))

observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[
    (df["InvoiceDate"] <= observation_end) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

df_pred = df[
    (df["InvoiceDate"] >= prediction_start) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]

# Build customer features only from observation window
df_obs["line_total"] = df_obs["Quantity"] * df_obs["UnitPrice"]

customer_temporal = df_obs.groupby("CustomerID").agg(
    total_revenue=("line_total","sum"),
    number_of_orders=("InvoiceNo","nunique"),
    distinct_products=("StockCode","nunique"),
    first_purchase=("InvoiceDate","min"),
    last_purchase=("InvoiceDate","max")
).reset_index()

customer_temporal["customer_lifetime_days"] = (
    customer_temporal["last_purchase"] - customer_temporal["first_purchase"]
).dt.days

# Create future target
target_future = df_pred.groupby("CustomerID")["InvoiceNo"].nunique().gt(0).astype(int)

customer_temporal = customer_temporal.merge(
    target_future.rename("target_purchase_future"),
    on="CustomerID",
    how="left"
)

customer_temporal["target_purchase_future"] = (
    customer_temporal["target_purchase_future"].fillna(0).astype(int)
)

X_temp = customer_temporal.drop(
    columns=["CustomerID","target_purchase_future","first_purchase","last_purchase"]
)
y_temp = customer_temporal["target_purchase_future"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

model_temp = LogisticRegression(max_iter=1000)
model_temp.fit(X_train_t, y_train_t)

y_pred_temp = model_temp.predict(X_test_t)

accuracy_temp = accuracy_score(y_test_t, y_pred_temp)
precision_temp = precision_score(y_test_t, y_pred_temp, zero_division=0)
recall_temp = recall_score(y_test_t, y_pred_temp, zero_division=0)
f1_temp = f1_score(y_test_t, y_pred_temp, zero_division=0)

print("\nTemporal split (correct model)")
print("Accuracy:", accuracy_temp)
print("Precision:", precision_temp)
print("Recall:", recall_temp)
print("F1:", f1_temp)

target        1.000000
Quantity      0.140300
InvoiceNo     0.107090
Revenue       0.098286
UnitPrice    -0.013961
CustomerID   -0.019600
Name: target, dtype: float64

Temporal split (correct model)
Accuracy: 0.665742024965326
Precision: 0.7157534246575342
Recall: 0.5694822888283378
F1: 0.6342943854324734


C:\Users\user\AppData\Local\Temp\ipykernel_1420\3696655195.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_obs["line_total"] = df_obs["Quantity"] * df_obs["UnitPrice"]
